In [1]:
# libraries
import os
import sys
import json
import hashlib
import platform
from pathlib import Path
import importlib.metadata
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw, ImageOps
from IPython.display import display

In [2]:
# globals; edit this cell for a new experiment, then rerun the whole pipeline
ROOT = Path(os.environ['NCA_PROJECT_ROOT'])
TASK = ROOT / 'a_import'
INPUT, OUTPUT = TASK / 'input', TASK / 'output'
SEED = 63
PROFILE = os.environ['NCA_PROFILE']
PROFILES = {
    'smoke': {'grid': 24, 'sprite': 10, 'shift': 2, 'state_dim': 8, 'parameter_target': 1200,
              'batch_size': 2, 'pool_size': 8, 'steps1': 4, 'steps2': 4, 'rollout_min': 4, 'rollout_max': 6,
              'eval_steps': 6, 'val_every': 2, 'replicates': 1, 'eval_cases': 3, 'latent_samples': 4,
              'channel_trials': 1, 'val_trials': 1, 'calibration_ps': [0.0, 0.5, 1.0]},
    'pilot': {'grid': 32, 'sprite': 14, 'shift': 3, 'state_dim': 16, 'parameter_target': 6000,
              'batch_size': 4, 'pool_size': 32, 'steps1': 600, 'steps2': 900, 'rollout_min': 24, 'rollout_max': 40,
              'eval_steps': 48, 'val_every': 100, 'replicates': 1, 'eval_cases': 12, 'latent_samples': 16,
              'channel_trials': 2, 'val_trials': 2, 'calibration_ps': [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]},
    'full': {'grid': 40, 'sprite': 18, 'shift': 4, 'state_dim': 16, 'parameter_target': 12000,
             'batch_size': 8, 'pool_size': 128, 'steps1': 4000, 'steps2': 6000, 'rollout_min': 48, 'rollout_max': 72,
             'eval_steps': 96, 'val_every': 200, 'replicates': 3, 'eval_cases': 32, 'latent_samples': 64,
             'channel_trials': 3, 'val_trials': 3, 'calibration_ps': [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]}
}
MODELS = [
    {'name': '1A_det', 'family': 'message', 'message_stochastic': False, 'query_stochastic': False, 'response_stochastic': False},
    {'name': '1A_bern', 'family': 'message', 'message_stochastic': True, 'query_stochastic': False, 'response_stochastic': False},
    {'name': '1B_det', 'family': 'query_answer', 'message_stochastic': False, 'query_stochastic': False, 'response_stochastic': False},
    {'name': '1B_qbern', 'family': 'query_answer', 'message_stochastic': False, 'query_stochastic': True, 'response_stochastic': False},
    {'name': '1B_rbern', 'family': 'query_answer', 'message_stochastic': False, 'query_stochastic': False, 'response_stochastic': True},
    {'name': '1B_qrbern', 'family': 'query_answer', 'message_stochastic': False, 'query_stochastic': True, 'response_stochastic': True}
]
requested_device = os.environ['NCA_DEVICE']
auto_device = 'cuda' if torch.cuda.is_available() else 'cpu'
CFG = {**PROFILES[PROFILE], 'profile': PROFILE, 'seed': SEED, 'models': MODELS,
       'device': auto_device if requested_device == 'auto' else requested_device,
       'resume': os.environ['NCA_RESUME'] == '1', 'cpu_threads': 1, 'cue_dim': 9,
       'budget_regimes': ['updates', 'bits'], 'learning_rate': 0.001,
       'step_size': 0.15, 'state_bound': 8.0, 'grad_clip': 1.0, 'state_penalty': 0.0001,
       'foreground_weight': 4.0, 'bernoulli_floor': 0.05, 'temperature': 1.0,
       'reset_fraction': 0.25, 'damage_probability': 0.25, 'switch_probability': 0.5,
       'damage_radius_fraction': 0.15, 'train_ps': [0.0, 0.25, 0.5, 0.75, 1.0],
       'stress_flip_rates': [0.0, 0.2] if PROFILE == 'smoke' else [0.0, 0.05, 0.1, 0.2, 0.35],
       'quality_ratio_max': 0.35, 'class_margin_min': 0.15, 'alpha_iou_min': 0.4,
       'stable_success_points': 2, 'phase2_initialization': 'phase1', 'protocol_version': 1}
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(CFG['cpu_threads'])
torch.use_deterministic_algorithms(True)

In [3]:
# source images; authored geometric fixtures, no font/network/download dependency
def fixture(kind):
    image = Image.new('RGBA', (128, 128), (0, 0, 0, 0))
    draw = ImageDraw.Draw(image)
    dark = (35, 31, 39, 255)
    if kind == 'cat':
        draw.polygon([(17, 58), (15, 9), (48, 34), (83, 34), (115, 9), (111, 61)], fill=(225, 143, 48, 255), outline=dark, width=4)
        draw.polygon([(23, 38), (22, 21), (39, 36)], fill=(246, 148, 156, 255))
        draw.polygon([(91, 36), (107, 21), (105, 39)], fill=(246, 148, 156, 255))
        draw.ellipse((14, 29, 115, 119), fill=(241, 174, 66, 255), outline=dark, width=4)
        draw.ellipse((27, 58, 50, 79), fill=(101, 158, 128, 255), outline=dark, width=3)
        draw.ellipse((80, 58, 103, 79), fill=(101, 158, 128, 255), outline=dark, width=3)
        draw.line((39, 62, 39, 74), fill=dark, width=4)
        draw.line((91, 62, 91, 74), fill=dark, width=4)
        draw.polygon([(57, 83), (73, 83), (65, 92)], fill=(178, 88, 107, 255))
        draw.arc((45, 87, 66, 104), 0, 160, fill=dark, width=3)
        draw.arc((65, 87, 86, 104), 20, 180, fill=dark, width=3)
        for y in [85, 94]:
            draw.line((7, y - 4, 43, y), fill=dark, width=3)
            draw.line((87, y, 123, y - 4), fill=dark, width=3)
    else:
        draw.ellipse((10, 10, 118, 118), fill=(250, 211, 65, 255), outline=dark, width=4)
        draw.ellipse((35, 41, 46, 59), fill=dark)
        draw.ellipse((82, 41, 93, 59), fill=dark)
        draw.arc((33, 50, 96, 100), 5, 175, fill=dark, width=7)
        draw.ellipse((23, 67, 38, 76), fill=(241, 144, 106, 255))
        draw.ellipse((90, 67, 105, 76), fill=(241, 144, 106, 255))
    return image

In [4]:
# import boundary; validate only here, then trust downstream inputs
assets = OUTPUT / 'assets'
assets.mkdir(exist_ok=True)
image_rows = []
for name in ['cat', 'smiley']:
    supplied = INPUT / f'{name}.png'
    image = Image.open(supplied).convert('RGBA') if supplied.is_file() else fixture(name)
    array = np.asarray(image)
    if not np.any(array[..., 3] > 0):
        raise ValueError(f'{name}.png has no foreground')
    image = ImageOps.contain(image, (CFG['sprite'], CFG['sprite']), Image.Resampling.LANCZOS)
    canvas = Image.new('RGBA', (CFG['grid'], CFG['grid']), (0, 0, 0, 0))
    corner = (CFG['grid'] // 2 - image.width // 2, CFG['grid'] // 2 - image.height // 2)
    canvas.alpha_composite(image, corner)
    canvas.save(assets / f'{name}.png')
    image_rows.append({'name': name, 'origin': 'provided' if supplied.is_file() else 'authored geometric fixture',
                       'sha256': hashlib.sha256((assets / f'{name}.png').read_bytes()).hexdigest()})
if CFG['grid'] <= 2 * CFG['shift'] + np.sqrt(2) * CFG['sprite']:
    raise ValueError('grid too small for the requested translated/rotated sprite')
if not 0 <= CFG['bernoulli_floor'] < 0.5:
    raise ValueError('Bernoulli floor must be in [0, 0.5)')
display(pd.DataFrame(image_rows))

,name,origin,sha256
0,cat,authored geometric fixture,e07caebac84c412e6192918f5e6e0121d5a616ce74e3f2...
1,smiley,authored geometric fixture,eec3a49281df456431b658de160b829fba2258f6be58a6...


In [5]:
# experimental contract; seed-only continuous cues, binary-only cell-to-cell access
design = pd.DataFrame([
    ('state', 'RGBA + private hidden channels; every grid site computes synchronously'),
    ('seed cue', 'one site only: presence, cos(theta), sin(theta), saturation, RGB gains, p, u'),
    ('translation', 'move the seed; no absolute coordinates enter the model'),
    ('rotation/color', 'conditioned reconstruction and held-out transformation generalization; not guaranteed group invariance'),
    ('messages', 'one hard bit per directed edge; direction-aware addressed inbox with eight Moore ports'),
    ('query/answer', 'requester sends q; neighbor receiver reads local state + q; neighbor sends one-bit r back'),
    ('no bypass', 'no neighbor RGBA/hidden reads, spatial convolutions, global cue broadcast or neighbor alpha gating'),
    ('hard training', 'hard threshold or Bernoulli in forward pass; biased sigmoid straight-through gradients'),
    ('Bernoulli', 'p_bit = floor + (1 - 2*floor)*sigmoid(logit / temperature); floor keeps uncertainty from vanishing'),
    ('external errors', 'independent bit flips are evaluation interventions, separate from Bernoulli generation'),
    ('budget regimes', 'same update counts or same directed-bit counts; parameter counts approximately matched'),
    ('Phase 2 choice', 'target = smiley iff u < p; only p and u enter the seed, never the chosen class'),
    ('response', 'change seed p/u in a running state; also test damage repair and persistence'),
    ('limits', 'two known templates; no claim of novel categories, autonomous evolution or VAE/GAN training'),
    ('replicates', 'all generators initialized at 63; disjoint jumped/advanced streams, paired across configurations'),
    ('validation', 'checkpoint selection uses validation only; hold-out tests never affect optimizer or selection')
], columns=['item', 'definition'])
design.to_csv(OUTPUT / 'experimental_contract.csv', index=False)
display(design)

,item,definition
0,state,RGBA + private hidden channels; every grid sit...
1,seed cue,"one site only: presence, cos(theta), sin(theta..."
2,translation,move the seed; no absolute coordinates enter t...
3,rotation/color,conditioned reconstruction and held-out transf...
4,messages,one hard bit per directed edge; direction-awar...
5,query/answer,requester sends q; neighbor receiver reads loc...
6,no bypass,"no neighbor RGBA/hidden reads, spatial convolu..."
7,hard training,hard threshold or Bernoulli in forward pass; b...
8,Bernoulli,p_bit = floor + (1 - 2*floor)*sigmoid(logit / ...
9,external errors,independent bit flips are evaluation intervent...


In [6]:
# provenance and configuration
sources = pd.DataFrame([
    ('NCA growth/persistence/repair', 'https://distill.pub/2020/growing-ca/'),
    ('local conditional response', 'https://arxiv.org/abs/2305.12971'),
    ('straight-through estimator caveat', 'https://arxiv.org/abs/1308.3432'),
    ('reproducibility limits', 'https://docs.pytorch.org/docs/stable/notes/randomness.html'),
    ('Bernoulli sampling API', 'https://docs.pytorch.org/docs/stable/generated/torch.bernoulli.html')
], columns=['topic', 'url'])
sources.to_csv(OUTPUT / 'sources.csv', index=False)
packages = ['torch', 'numpy', 'pandas', 'seaborn', 'matplotlib', 'pillow', 'nbformat', 'nbclient', 'ipykernel', 'scipy']
environment = {name: importlib.metadata.version(name) for name in packages}
environment.update({'python': sys.version, 'platform': platform.platform(), 'device': CFG['device']})
(OUTPUT / 'environment.json').write_text(json.dumps(environment, indent=2))
(OUTPUT / 'image_provenance.json').write_text(json.dumps(image_rows, indent=2))
(OUTPUT / 'config.json').write_text(json.dumps(CFG, indent=2))
display(pd.DataFrame([{'profile': PROFILE, 'device': CFG['device'], 'seed': SEED,
                       'training_runs': 12 * len(CFG['budget_regimes']) * CFG['replicates']}]))

,profile,device,seed,training_runs
0,smoke,cpu,63,24
